In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 73.8 MB/s eta 0:00:00


## Imports

In [2]:
import json
import fitz  # PyMuPDF
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
#from transformers import BitsAndBytesConfig

## Loading the document

In [3]:
pdf_path="/kaggle/input/datasets/koushikikundu/hr-policies/HR Policy Manual 2023 (8).pdf"

doc = fitz.open(pdf_path)

In [4]:
pages = []

for i, page in enumerate(doc):
    # Find tables on the current page
    tabs = page.find_tables()
    
    if tabs.tables:
        # Extract table data structured as lists/dataframes
        extracted_tables = [tab.extract() for tab in tabs]
        
        # Get bounding boxes of all detected tables
        table_bboxes = [tab.bbox for tab in tabs]
        
        # Extract page text while ignoring text inside table bounding boxes
        text_page = page.get_text("words")  # list of (x0, y0, x1, y1, word, block_no, line_no, word_no)
        non_table_words = []
        
        for word_info in text_page:
            word_bbox = fitz.Rect(word_info[:4])
            # Check if word falls inside any table bbox
            in_table = any(word_bbox.intersects(tbl_box) for tbl_box in table_bboxes)
            if not in_table:
                non_table_words.append(word_info[4])
        
        page_text = " ".join(non_table_words)
        
        pages.append({
            "page": i + 1,
            "text": page_text.strip(),
            "tables": extracted_tables
        })
    else:
        pages.append({
            "page": i + 1,
            "text": page.get_text("text").strip()
        })

print("Total pages:", len(pages))

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Total pages: 208


In [5]:
print(pages[107]['tables'])

[[['CATEGORIES FOR GSLI', None, 'Sum Assured'], ['Category “A”', 'Faculty, Staff in Pay Level 9 and above', '2,80,000/-'], ['Category “B”', 'Staff in Pay Level 5 to Pay Level 8', '2,25,000/-'], ['Category “C”', 'Staff in Pay Level 2 to Pay Level 4', '1,40,000/-'], ['Category “D”', 'Staff in Pay Level 1', '70,000/-']]]


In [6]:
pages=pages[10:37]+pages[39:41]+pages[54:86]+pages[94:115]+pages[116:123]+pages[124:138]+pages[152:164]+pages[165:176]+pages[177:194]

## Preprocessing 

In [7]:
def normalize(s):
    s = s.replace("\xa0", " ")
    s = re.sub(r"Page\s+\d+", " ", s)
    s=s.replace("IIMA HR Policy Manual 2023", "")
    s = re.sub(r"\n+", "\n",s)
    s= re.sub(r" +", " ", s)
    return s.strip()
    

for page in pages:
    text = normalize(page["text"])   

    if "tables" in page:

        cleaned_tables = []
        for table in page["tables"]:

            cleaned_table = []

            for row in table:

                cleaned_row = []
                for cell in row:

                    cell = "" if cell is None else cell.strip()

                    # Clean the cell
                    cell = normalize(cell)

                    cleaned_row.append(cell)

                cleaned_table.append(cleaned_row)

            cleaned_tables.append(cleaned_table)

        page["tables"] = cleaned_tables

    page["text"] = text

In [8]:
def tables_to_text(tables):
    if not tables:
        return ""

    output = []

    for idx, table in enumerate(tables, start=1):
        output.append(f"\nTable {idx}")

        for row in table:
            row = [
                str(cell).replace("\n", " ").strip()
                for cell in row
                if cell is not None and str(cell).strip() != ""
            ]

            if row:
                output.append(" | ".join(row))

    return "\n".join(output)

In [9]:
page_contents = []

for page in pages:

    page_text = page["text"].strip()
    if 'tables' in page:

        table_text = tables_to_text(page["tables"])

        combined = page_text + "\n\n" + table_text
    else:
        combined = page_text

    page_contents.append(combined)

In [10]:
page_contents[5]

'6\n\n\nTable 1\n8. | Prof. Ajay Pandey IIM Ahmedabad | Chairman’s Nominee\n9. | Prof. Sachin Jayaswal IIM Ahmedabad | Chairman’s Nominee\n10. | Ramesh Mangaleswaran Senior Partner Emeritus, McKinsey & Company Chennai, Tamil Nadu, India | Co-opted by the Board from the Alumni\n11. | Dr. Hasit Joshipura Advisor to L&T Group CEO and MD, Data Centre & Cloud, Innovation Fund Larsen & Toubro Limited Landmark A Wing, 5th Floor, Suren Road Off. Andheri-Kurla Road Andheri (East), Mumbai – 400093 | -do-\n12. | Rama Bijapurkar 206, Nirman Kendra, Dr. E. Moses Road, Mahalakshmi, Mumbai 400 011. | -do-\n13. | Prof. Pradeep K. Chintagunta Joseph T. and Bernice S. Lewis Distinguished Service Professor of Marketing University of Chicago Booth School of Business Chicago, IL 60637, USA | -do-\n14. | Samir U. Mehta Chairman, Torrent Group Torrent House, Off. Ashram Road, Ahmedabad – 380009 | Co-opted by the Board from the members of Society\n15. | Prof. Errol D’Souza Director IIM Ahmedabad | Ex-Officio\

## Creating the dataset

In [11]:
def make_chunks(page_contents,
                pages_per_chunk=3,
                overlap=1):

    chunks = []

    step = pages_per_chunk - overlap

    for start in range(0, len(page_contents), step):

        chunk = "\n\n".join(
            page_contents[start:start+pages_per_chunk]
        )

        chunks.append(chunk)

    return chunks

In [12]:
chunks = make_chunks(
    page_contents,
    pages_per_chunk=3,
    overlap=1
)

In [13]:
len(chunks)

72

In [14]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).cuda()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [15]:
def create_prompt(chunk):

    return f"""
You are an expert HR policy dataset generator.

Your task is to create training Question-Answer pairs for an HR policy
chatbot using ONLY the information contained in the policy content below.

The chatbot will be used by employees to ask questions about:

- Leave
- Employee benefits
- Compensation
- Insurance
- Retirement
- Working hours
- Attendance
- Holidays
- Eligibility
- Recruitment
- Performance
- Promotions
- Training
- Employee procedures
- Other HR policies

============================================================
STRICT INFORMATION RULES
============================================================

1. Use ONLY information explicitly present in the policy content.

2. Never invent information.

3. Never use outside knowledge.

4. Preserve exact numerical values.

5. Preserve exact monetary amounts.

6. Preserve percentages.

7. Preserve dates.

8. Preserve durations and time periods.

9. Preserve eligibility conditions.

10. Preserve restrictions.

11. Preserve exceptions.

12. Preserve procedures.

13. Preserve table information.

14. Never replace an exact value with a vague description.

Example:

Source:
"Employees are entitled to 15 days of casual leave per year."

Bad:
"Employees are entitled to casual leave."

Good:
"Employees are entitled to 15 days of casual leave per year."

============================================================
TABLE RULES
============================================================

If the source contains a table:

- Read the table carefully.
- Preserve important rows and columns.
- Preserve exact values.
- Preserve relationships between columns.
- Do not invent missing values.

If the table describes employee benefits, leave, compensation,
eligibility, coverage, limits, or other HR policies, include that
information in the generated answers.

When a table is useful for answering a question, use a Markdown table.

Example:

| Benefit | Eligibility | Coverage |
|---|---|---|
| Medical Insurance | Employees | ₹5,00,000 |
| Life Insurance | Employees | ₹10,00,000 |

============================================================
QUESTION PRIORITY
============================================================

Prioritize HR POLICY information.

Priority order:

1. Employee benefits
2. Leave policies
3. Numerical entitlements
4. Compensation
5. Insurance
6. Eligibility
7. Conditions
8. Exceptions
9. Procedures
10. Working conditions
11. Other HR rules

General institutional information such as:

- Board members
- Personal addresses
- Names of directors
- Names of government nominees
- Company addresses

should NOT be turned into QA pairs unless it is directly relevant
to an HR policy.

============================================================
QUESTION TYPES
============================================================

Use a mixture of:

1. Direct factual questions

2. Numerical questions

3. Scenario-based questions

4. Calculation questions

5. Eligibility questions

6. Condition/restriction questions

7. Procedure questions

8. Table-based questions

9. Comparison questions

============================================================
NUMERICAL QUESTIONS
============================================================

If the source contains a numerical HR rule, create at least one
question that tests the number.

Example:

Source:
"Employees receive 15 casual leave days per year."

Question:
"I have already taken 10 casual leave days. How many do I have left?"

Answer:
"Employees receive 15 casual leave days per year. After taking
10 days, 5 days remain.

Calculation: 15 - 10 = 5 days."

Only perform calculations using numbers explicitly given in the source.

============================================================
DUPLICATION RULE
============================================================

DO NOT repeat questions.

Every question must be meaningfully different.

Do not ask the same question using slightly different wording.

For example, these should NOT both appear:

"What is the casual leave entitlement?"

"How many casual leave days are provided?"

Choose only one.

============================================================
NUMBER OF QUESTIONS
============================================================

Generate EXACTLY 8-12 Question-Answer pairs as per need.

However, if the policy content contains fewer than 8 meaningful
HR-policy facts, generate only as many as can be supported.

Do NOT create questions about information that is not useful for
an employee simply to reach 8 questions.

============================================================
FINAL QUALITY CHECK
============================================================

Before returning the answer, verify:

- No duplicate questions.
- No invented information.
- Exact numbers preserved.
- Exact amounts preserved.
- Percentages preserved.
- Important dates preserved.
- Eligibility preserved.
- Conditions preserved.
- Exceptions preserved.
- Important table information preserved.
- Calculation questions use only source numbers.
- Answers are complete.
- Questions are useful to employees.
- General institutional information is not prioritized over HR policies.

============================================================
OUTPUT
============================================================

Return ONLY valid JSON.

Do not use Markdown code fences.

Return exactly this structure:

[
  {{
    "question": "...",
    "answer": "..."
  }},
  {{
    "question": "...",
    "answer": "..."
  }}
]

============================================================
POLICY CONTENT
============================================================

{chunk}
"""

In [16]:
def parse_json_response(response):

    response = response.strip()

    # Remove accidental markdown code fences
    response = re.sub(r"^```json\s*", "", response)
    response = re.sub(r"^```\s*", "", response)
    response = re.sub(r"\s*```$", "", response)

    try:
        data = json.loads(response)

        if isinstance(data, list):
            return data

    except json.JSONDecodeError:
        return None

    return None



In [17]:
def validate_qa(qa):

    if not isinstance(qa, list):
        return []

    valid = []
    seen_questions = set()

    for item in qa:

        if not isinstance(item, dict):
            continue

        question = item.get("question")
        answer = item.get("answer")

        if not isinstance(question, str):
            continue

        if not isinstance(answer, str):
            continue

        question = question.strip()
        answer = answer.strip()

        if not question or not answer:
            continue

        # Remove duplicate questions
        question_key = question.lower().strip()

        if question_key in seen_questions:
            continue

        seen_questions.add(question_key)

        valid.append({
            "question": question,
            "answer": answer
        })

    return valid

In [18]:
dataset = []

output_file = "hr_policy_qa.jsonl"

# Start fresh
with open(output_file, "w", encoding="utf-8") as f:
    pass


for i, chunk in enumerate(chunks):

    print("=" * 80)
    print(f"Processing chunk {i + 1}/{len(chunks)}")

    prompt = create_prompt(chunk)

    messages = [
        {
            "role": "system",
            "content": (
                "You generate accurate HR policy training data. "
                "Never invent policy information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=3000,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    qa = parse_json_response(response)

    if qa is None:

        print(f"Chunk {i + 1}: Invalid JSON")

        with open(
            f"failed_chunk_{i + 1}.txt",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(response)

        continue

    qa = validate_qa(qa)

    # Safety check
    if len(qa) > 12:
        print(
            f"Chunk {i + 1}: {len(qa)} pairs generated. "
            f"Keeping first 12."
        )

        qa = qa[:12]

    if len(qa) == 0:

        print(f"Chunk {i + 1}: No valid QA pairs")

        continue

    with open(
        output_file,
        "a",
        encoding="utf-8"
    ) as f:

        for item in qa:

            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                ) + "\n"
            )

            dataset.append(item)

    print(
        f"Chunk {i + 1}: Successfully added "
        f"{len(qa)} QA pairs"
    )


print("=" * 80)
print("DONE")
print(f"Total QA pairs: {len(dataset)}")
print(f"Saved to: {output_file}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processing chunk 1/72
Chunk 1: Successfully added 8 QA pairs
Processing chunk 2/72
Chunk 2: Successfully added 8 QA pairs
Processing chunk 3/72
Chunk 3: Successfully added 12 QA pairs
Processing chunk 4/72
Chunk 4: Successfully added 8 QA pairs
Processing chunk 5/72
Chunk 5: Successfully added 11 QA pairs
Processing chunk 6/72
Chunk 6: Successfully added 9 QA pairs
Processing chunk 7/72
Chunk 7: Successfully added 8 QA pairs
Processing chunk 8/72
Chunk 8: Successfully added 8 QA pairs
Processing chunk 9/72
Chunk 9: Successfully added 9 QA pairs
Processing chunk 10/72
Chunk 10: Successfully added 9 QA pairs
Processing chunk 11/72
Chunk 11: Successfully added 8 QA pairs
Processing chunk 12/72
Chunk 12: Successfully added 11 QA pairs
Processing chunk 13/72
Chunk 13: Successfully added 8 QA pairs
Processing chunk 14/72
Chunk 14: Successfully added 12 QA pairs
Processing chunk 15/72
Chunk 15: Successfully added 8 QA pairs
Processing chunk 16/72
Chunk 16: Successfully added 8 QA pairs
Proces

In [19]:
print(dataset[200:202])

[{'question': 'How much subsistence allowance is paid to an employee during a departmental enquiry lasting more than ninety days?', 'answer': 'For such period, the subsistence allowance shall for such period be equal to three-fourths of such basic salary, dearness allowance and other compensatory allowance.'}, {'question': "What is the subsistence allowance for an employee under an outside agency's enquiry lasting more than one hundred and eighty days?", 'answer': 'For such period, the subsistence allowance shall for such period be equal to three-fourths of such wage.'}]


## Creating the validation set

In [20]:
chunks_v = make_chunks(
    page_contents,
    pages_per_chunk=5,
    overlap=1
)

In [21]:
len(chunks_v)

36

In [22]:
def create_prompt_for_validation(chunk):

    return f"""
You are an expert HR policy evaluator.

Generate exactly 2 high-quality Question-Answer pairs from the HR policy below
for evaluating a fine-tuned HR chatbot.

RULES:
- Use ONLY information explicitly present in the policy.
- Do not use outside knowledge or assumptions.
- Do not invent facts, numbers, conditions, eligibility, procedures, or exceptions.
- Questions must be realistic employee HR questions.
- Do not copy sentences or headings directly.
- Questions must test understanding, not just keyword matching.
- Make the two questions test DIFFERENT information.
- Prefer questions involving rules, eligibility, conditions, exceptions, limits,
  procedures, time periods, tables, or numerical calculations when available.
- If the policy contains numbers, consider using them in a scenario/application
  question when possible.
- For calculations, use ONLY numbers explicitly stated in the policy.
- Answers must be precise, complete, and fully supported by the policy.
- Include relevant conditions, exceptions, and limits.
- Do not mention the policy chunk or provided text.

IMPORTANT:
The policy below is the ONLY source of truth. Do not assume information
outside it.

OUTPUT:
Return ONLY valid JSON.
No Markdown, no ```json, and no explanations.

Format:
[
    {{
        "question": "...",
        "answer": "..."
    }},
    {{
        "question": "...",
        "answer": "..."
    }}
]

HR POLICY:
{chunk}
"""

In [23]:
validation_dataset = []

# Start with a fresh file
open("hr_policy_qa_validation.jsonl", "w", encoding="utf-8").close()

for i, chunk in enumerate(chunks_v):

    print(f"\nProcessing chunk {i+1}/{len(chunks_v)}")

    prompt = create_prompt_for_validation(chunk)

    messages = [
        {
            "role": "system",
            "content": "You create validation datasets for HR chatbots."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=700,
        temperature=0.2,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        qa = json.loads(response)

        # Make sure the model actually returned a list
        if not isinstance(qa, list):
            raise ValueError("Output is not a JSON list")

        # Make sure there are 2 Q&A pairs
        if len(qa) != 2:
            raise ValueError(f"Expected 2 Q&A pairs, got {len(qa)}")

        # Validate structure
        for item in qa:
            if not isinstance(item, dict):
                raise ValueError("QA item is not an object")

            if set(item.keys()) != {"question", "answer"}:
                raise ValueError("Invalid fields in QA object")

            if not item["question"].strip():
                raise ValueError("Empty question")

            if not item["answer"].strip():
                raise ValueError("Empty answer")

        # Save immediately
        with open(
            "hr_policy_qa_validation.jsonl",
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(qa, ensure_ascii=False) + "\n"
            )

        validation_dataset.extend(qa)

        print(f"✓ Chunk {i+1} inserted")

    except Exception as e:

        print(f"✗ Chunk {i+1} failed")
        print("Model output:")
        print(response)
        print("Error:", e)


Processing chunk 1/36
✓ Chunk 1 inserted

Processing chunk 2/36
✓ Chunk 2 inserted

Processing chunk 3/36
✓ Chunk 3 inserted

Processing chunk 4/36
✓ Chunk 4 inserted

Processing chunk 5/36
✓ Chunk 5 inserted

Processing chunk 6/36
✓ Chunk 6 inserted

Processing chunk 7/36
✓ Chunk 7 inserted

Processing chunk 8/36
✓ Chunk 8 inserted

Processing chunk 9/36
✓ Chunk 9 inserted

Processing chunk 10/36
✓ Chunk 10 inserted

Processing chunk 11/36
✓ Chunk 11 inserted

Processing chunk 12/36
✓ Chunk 12 inserted

Processing chunk 13/36
✓ Chunk 13 inserted

Processing chunk 14/36
✓ Chunk 14 inserted

Processing chunk 15/36
✓ Chunk 15 inserted

Processing chunk 16/36
✓ Chunk 16 inserted

Processing chunk 17/36
✓ Chunk 17 inserted

Processing chunk 18/36
✓ Chunk 18 inserted

Processing chunk 19/36
✓ Chunk 19 inserted

Processing chunk 20/36
✓ Chunk 20 inserted

Processing chunk 21/36
✓ Chunk 21 inserted

Processing chunk 22/36
✓ Chunk 22 inserted

Processing chunk 23/36
✓ Chunk 23 inserted

Proce

In [24]:
len(validation_dataset)

72

In [25]:
dataset[0]

{'question': 'What is the duration of the MBA programme?',
 'answer': 'The Two-year Post Graduate Programme in Management (MBA) is a two-year programme.'}

In [26]:
validation_dataset[0]

{'question': 'What is the minimum number of faculty members that IIMA had in the 1970s?',
 'answer': 'The minimum number of faculty members that IIMA had in the 1970s was 65.'}